# Phase 3 lesion/vessel training (Kaggle, GPU)

Trains the vessel U-Net (DRIVE + CHASE_DB1 + STARE), the 4-class lesion U-Net (IDRiD
segmentation), and the microaneurysm patch classifier (IDRiD MA masks).

**Note on why this notebook exists at all:** these datasets are small (~88 vessel
images, 81 IDRiD segmentation images), so unlike Phase 2's grading model, this repo's
Phase 3 checkpoints were actually trained locally on the Apple M4 (MPS) -- the
datasets are light enough that this doesn't strain the 8GB local budget the way full
APTOS training would. This notebook exists for reproducibility and for scaling up if
larger/additional vessel or lesion datasets get added later.

**Setup before running:**
1. Attach four datasets via "+ Add Input": a DRIVE mirror (e.g.
   `andrewmvd/drive-digital-retinal-images-for-vessel-extraction`), a CHASE_DB1 mirror
   (e.g. `buffyhridoy/chase-db1`), a STARE mirror with ground truth (e.g.
   `vasavigneswar/stare-retinal-dataset` -- many STARE mirrors have raw images only, no
   ground truth; verify `labels-ah/` exists), and an IDRiD mirror (e.g.
   `aaryapatel98/indian-diabetic-retinopathy-image-dataset` -- verify it has the full
   `A. Segmentation` + `C. Localization` structure, not just grading images).
2. Turn on a GPU accelerator and internet access (same as the grading notebook).

In [ ]:
!pip install -q timm albumentations segmentation-models-pytorch onnx onnxscript

In [ ]:
import os

REPO_URL = "https://github.com/prithvipm412/SugarEyes.git"
REPO_DIR = "/kaggle/working/SugarEyes"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# Normalize whichever datasets are attached under /kaggle/input into this
# repo's data/raw/ layout, reusing the exact same processors the local
# download scripts use (see scripts/download_vessels.py, scripts/download_idrid.py)
# -- just pointed at /kaggle/input instead of a freshly-downloaded temp dir.
from pathlib import Path

from scripts.download_vessels import process_chasedb1, process_drive, process_stare
from scripts.download_idrid import process_localization, process_segmentation

KAGGLE_INPUT = Path("/kaggle/input")
input_dirs = {p.name: p for p in KAGGLE_INPUT.iterdir() if p.is_dir()}
print("attached datasets:", list(input_dirs))


def find_by_marker(marker_glob: str):
    for p in input_dirs.values():
        if list(p.rglob(marker_glob)):
            return p
    raise FileNotFoundError(f"No attached dataset matches marker {marker_glob!r}")


drive_root = find_by_marker("1st_manual")
chase_root = find_by_marker("*_1stHO.png")
stare_root = find_by_marker("*.ah.ppm")
idrid_root = find_by_marker("A. Segmentation")

n_drive = process_drive(drive_root, Path("data/raw/vessels/drive"))
n_chase = process_chasedb1(chase_root, Path("data/raw/vessels/chasedb1"))
n_stare = process_stare(stare_root, Path("data/raw/vessels/stare"))
seg_df = process_segmentation(idrid_root, Path("data/raw/idrid/segmentation"))
loc_df = process_localization(idrid_root, Path("data/raw/idrid/localization"))
print(f"drive={n_drive} chase={n_chase} stare={n_stare} idrid_seg={len(seg_df)} idrid_loc={len(loc_df)}")

In [ ]:
!python -m src.drscreen.lesions.vessels --config configs/vessels.yaml

In [ ]:
!python -m src.drscreen.lesions.unet --config configs/lesion_unet.yaml

In [ ]:
!python -m src.drscreen.lesions.candidates --config configs/ma_classifier.yaml

In [ ]:
# Export all three to ONNX (opset 13, legacy exporter -- see
# src/drscreen/grading/model.py's export_onnx for why dynamo=False is needed).
import torch
import yaml

from src.drscreen.lesions.candidates import MAPatchClassifier
from src.drscreen.lesions.candidates import PATCH_SIZE as MA_PATCH_SIZE
from src.drscreen.lesions.unet import LesionUNet
from src.drscreen.lesions.unet import PATCH_SIZE as LESION_PATCH_SIZE
from src.drscreen.lesions.vessels import VesselModel
from src.drscreen.lesions.vessels import PATCH_SIZE as VESSEL_PATCH_SIZE


def export(model, ckpt_path, size, channels, out_path):
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    dummy = torch.randn(1, channels, size, size)
    torch.onnx.export(model, dummy, out_path, input_names=["image"], output_names=["logits"],
                       dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, opset_version=13, dynamo=False)
    print(f"wrote {out_path}")


with open("configs/vessels.yaml") as f:
    vc = yaml.safe_load(f)
export(VesselModel(encoder_name=vc["encoder"], pretrained=False), "models/vessels/best.pt", VESSEL_PATCH_SIZE, 3, "models/vessels.onnx")

with open("configs/lesion_unet.yaml") as f:
    lc = yaml.safe_load(f)
export(LesionUNet(encoder_name=lc["encoder"], pretrained=False), "models/lesion_unet/best.pt", LESION_PATCH_SIZE, 3, "models/lesion_unet.onnx")

export(MAPatchClassifier(), "models/ma_classifier/best.pt", MA_PATCH_SIZE, 3, "models/ma_classifier.onnx")

## After this notebook finishes

Download `models/vessels/best.pt`, `models/lesion_unet/best.pt`, `models/ma_classifier/best.pt`
and their `.onnx` exports from the Output tab, and place them at the same paths locally.
Then run `python -m tests.gate_phase3` to verify against the checkpoints trained here.